Silver Layer (Cleaned Data): 
The Silver layer is where we start working with the data in a meaningful way. Data from the Bronze layer is cleaned, standardized, and structured so it becomes easier to analyze. This includes fixing data types, removing duplicates, handling missing values, and applying basic quality checks.
The goal here is to turn raw data into something reliable and consistent that can actually be used for reporting or further analysis.

#### Loading Data

In [0]:
from pyspark.sql.functions import *

orders_df = spark.table('workspace.bronze.orders')
customer_df = spark.table('workspace.bronze.customers')
order_items_df = spark.table('workspace.bronze.order_items')
products_df = spark.table('workspace.bronze.products')
sellers_df = spark.table('workspace.bronze.sellers')
pcat_df = spark.table('workspace.bronze.product_category_name_translation')
order_review_df = spark.table('workspace.bronze.order_reviews')
payment_df = spark.table('workspace.bronze.order_payments')
location_df = spark.table('workspace.bronze.geolocation')


#### Verifying Schemas

In [0]:
orders_df.printSchema()
customer_df.printSchema()
order_items_df.printSchema()
products_df.printSchema()
sellers_df.printSchema()
pcat_df.printSchema()
order_review_df.printSchema()
location_df.printSchema()

#### Data Type Correction

In [0]:
products_df = products_df\
    .withColumn('product_description_length',col('product_description_length').cast("int"))\
    .withColumn('product_photos_qty',col('product_photos_qty').cast("int"))\
    .withColumn('product_name_length',col('product_name_length').cast("int"))

#### Handling null values

Orders DF
Business Logic:\
1.Remove null values where delivery status is "delivered" and there is no delivered date.\
2.Keep other null values as it is.


In [0]:
print(orders_df.count())

orders_df = orders_df.filter(~(\
  (col('order_status') == "delivered") & 
  (col('order_delivered_customer_date').isNull()))
                             )
print(orders_df.count())

Products DF Business Logic:\
1.Fill null values with 'uncategorized' for product category name.\
2.Fill null values with median for product weight,length, height and width. (For getting proper distribution).

In [0]:
products_df = products_df.fillna('uncategorized', subset = 'product_category_name')

cols = ['product_weight_gm','product_length_cm','product_height_cm','product_width_cm']
for c in cols:
    median_val = products_df.approxQuantile(c,[0.5],0)
    products_df = products_df.fillna(median_val[0], subset = c)

Review DF Business Logic:\
1.Fill review comment title with 'No title'\
2.Fill review comment message with 'No review'

In [0]:
order_review_df = order_review_df.fillna({
    'review_comment_title':'No title',
    'review_comment_message':'No review'
})

order_review_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in order_review_df.columns]).show()

#### Data Standardization
For city names there some different accents, eg: sao paulo -> 'são paulo',etc

In [0]:
import unicodedata
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def to_ascii(text):
    if text:
        return unicodedata.normalize("NFKD", text)\
            .encode("ascii", "ignore")\
            .decode("utf-8").replace("-"," ")
    return text

to_ascii_udf = udf(to_ascii, StringType())

In [0]:
location_df.filter(
    col("geolocation_city").rlike(r"[^\x00-\x7F]")
).select("geolocation_city").distinct().show(2085,False)

In [0]:
location_df = location_df.withColumn("geolocation_city", to_ascii_udf(col("geolocation_city")))

In [0]:
location_df.filter(col('geolocation_city').rlike(r'[^\x00-\x7F]')).show()

In [0]:
customer_df.filter(
    col("customer_city").rlike(r"[^\x00-\x7F]")
).select("customer_city").distinct().show(2085,False)

In [0]:
sellers_df.filter(
    col("seller_city").rlike(r"[^\x00-\x7F]")
).select("seller_city").distinct().show(2085,False)

In [0]:
sellers_df = sellers_df.withColumn("seller_city", to_ascii_udf(col("seller_city")))
sellers_df.filter(
    col("seller_city").rlike(r"[^\x00-\x7F]")
).select("seller_city").distinct().show(2085,False)

Business Logic:\
1.Remove leading/trailing spaces from string type columns/features.\
2.Convert city names to lower case and state names to upper case.\
3.Convert order status(orders_df) and payment type(payment_df) to lower case.\
4.Convert product category name to lower case.

In [0]:
orders_df = orders_df\
    .withColumn('order_status', lower(trim(col('order_status')))) \
    .withColumn('order_id', trim(col('order_id'))) \
    .withColumn('customer_id', trim(col('customer_id')))

customer_df = customer_df\
    .withColumn('customer_id', trim(col('customer_id')))\
    .withColumn('customer_unique_id', trim(col('customer_unique_id')))\
    .withColumn('customer_city',lower(trim(col('customer_city'))))\
    .withColumn('customer_state',upper(trim(col('customer_state'))))

order_items_df = order_items_df\
    .withColumn('order_id', trim(col('order_id')))\
    .withColumn('product_id', trim(col('product_id')))\
    .withColumn('seller_id', trim(col('seller_id')))

products_df = products_df\
    .withColumn('product_id',trim(col('product_id')))\
    .withColumn('product_category_name',lower(trim(col('product_category_name'))))\

pcat_df = pcat_df\
    .withColumn('product_category_name',lower(trim(col("product_category_name"))))\
    .withColumn('product_category_name_english',lower(trim(col("product_category_name_english"))))

sellers_df = sellers_df\
    .withColumn("seller_id",trim(col("seller_id")))\
    .withColumn("seller_city",lower(trim(col("seller_city"))))\
    .withColumn('seller_state',upper(trim(col("seller_state"))))

location_df = location_df\
    .withColumn('geolocation_city',lower(trim(col("geolocation_city"))))\
    .withColumn('geolocation_state',upper(trim(col("geolocation_state"))))

payment_df = payment_df\
    .withColumn('order_id',trim(col("order_id")))\
    .withColumn('payment_type',lower(trim(col("payment_type"))))

order_review_df = order_review_df\
    .withColumn('review_id',trim(col("review_id")))\
    .withColumn('order_id',trim(col("order_id")))

#### Derived Columns

Business Logic:\
1.orders_df : Add a column 'delivery_days' -> purchase_date - delivery date\
2.order_items_df : Add a column 'total_value' -> price + freight_value

In [0]:
orders_df = orders_df.withColumn('delivery_days',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))
display(orders_df)

In [0]:
order_items_df = order_items_df.withColumn('total_price', round(col('price') + col('freight_value'),2))
display(order_items_df)

#### Joins

Joining products_df and pcat_df on key product_category_name

In [0]:
#Selecting only required columns from products
products_df = products_df.select('product_id','product_category_name',
                                 'product_weight_gm','product_length_cm','product_height_cm','product_width_cm')

In [0]:
final_products_df = products_df.join(pcat_df,on ="product_category_name", how='left')
final_products_df.columns

In [0]:
final_products_df = final_products_df\
    .withColumn('product_category_name_english', when(col('product_category_name') == 'uncategorized','uncategorized').otherwise(col('product_category_name_english')))\
    .withColumn('product_category_name_english',when(col("product_category_name") == 'pc_gamer','pc_gaming').otherwise(col("product_category_name_english")))\
    .withColumn('product_category_name_english',when(col("product_category_name") == 'portateis_cozinha_e_preparadores_de_alimentos','portable_kitchen_and_food_preparators').otherwise(col("product_category_name_english")))

In [0]:
final_products_df = final_products_df.withColumn('product_category_name',col('product_category_name_english')).drop('product_category_name_english')

In [0]:
display(final_products_df)

In [0]:
final_products_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in final_products_df.columns]).show()

In [0]:
final_products_df.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.products')

Joining customer_df with location_df on zipcode


In [0]:
#Aggregating longitude and lat
clean_location_df = location_df.groupBy('geolocation_zipcode_prefix').agg(
    avg('geolocation_lat').alias('geolocation_lat'),
    avg('geolocation_lng').alias('geolocation_lng'),
    first('geolocation_city').alias('geolocation_city'),
    first('geolocation_state').alias('geolocation_state')
)

In [0]:
clean_location_df.groupBy("geolocation_zipcode_prefix").count().filter(col("count") > 1).show()

In [0]:
cust_geo_df = customer_df.join(clean_location_df, 
                               on = customer_df.customer_zipcode_prefix == clean_location_df.geolocation_zipcode_prefix,
                               how = "left").drop('geolocation_zipcode_prefix')
display(cust_geo_df)


In [0]:
cust_geo_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in cust_geo_df.columns]).show()

In [0]:
cust_geo_df = cust_geo_df.withColumnRenamed('customer_zipcode_prefix','zipcode')

In [0]:
cust_geo_df.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.customers')

Joining sellers_df to location_df on zipcode

In [0]:
clean_location_df.columns

In [0]:
seller_geo_df = sellers_df.join(clean_location_df,
                                on = sellers_df.seller_zipcode_prefix == clean_location_df.geolocation_zipcode_prefix,
                                how = "left").drop('geolocation_zipcode_prefix')
display(seller_geo_df)

In [0]:
seller_geo_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in seller_geo_df.columns]).show()

In [0]:
seller_geo_df = seller_geo_df.withColumnRenamed('seller_zipcode_prefix','zipcode')

In [0]:
seller_geo_df.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.sellers')

Joining final_products_df to order_items_df by product_id key

In [0]:
order_items_final_df = order_items_df.join(final_products_df, on = "product_id",
                                           how = 'left').drop(final_products_df.product_id)
order_items_final_df.display()

In [0]:
order_items_final_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in order_items_final_df.columns]).show()

In [0]:
order_items_final_df.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.order_items')

In [0]:
payment_df = payment_df.select('order_id','payment_value','payment_type','payment_installments')
payment_df.columns
payment_df.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.order_payments')

In [0]:
order_review_df.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.order_reviews')

In [0]:
orders_df.write.format('delta').mode('overwrite').saveAsTable('workspace.silver.orders')